In [ ]:
# Install required libraries
!pip install bert-score rouge-score pandas numpy transformers accelerate

# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import re
import pandas as pd
import numpy as np
from bert_score import score

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FOLDER = "/content/drive/MyDrive/Research/BVQA Dataset/BVQA_Master/3_Annotations/Performance Records/Results/On Test Split/fix"
OUTPUT_FOLDER = "/content/drive/MyDrive/Research/BVQA Dataset/BVQA_Master/3_Annotations/Performance Records/Performance Matrices/On Test Split/fix"

# Ensure output directory exists
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Find all CSV files in the input folder and its subfolders recursively
csv_files = glob.glob(os.path.join(INPUT_FOLDER, "**/*.csv"), recursive=True)

print("==========================================")
print(f"Found {len(csv_files)} model result file(s) to process:")
print("==========================================")
for idx, file_path in enumerate(csv_files, start=1):
    relative_path = os.path.relpath(file_path, INPUT_FOLDER)
    print(f"{idx}. {relative_path}")
print("==========================================")

# --- USER CONFIRMATION STEP ---
user_input = input("Do you want to proceed with the calculation? Type 'yes' to start: ").strip().lower()

if user_input != 'yes':
    print("\nCalculation cancelled by user. Terminating process.")
else:
    print("\nConfirmation received. Starting calculations...\n")

    # ==========================================
    # 1. BENGALI TEXT NORMALIZATION FUNCTION
    # ==========================================
    def normalize_text(text):
        if not isinstance(text, str):
            text = str(text)

        # Lowercase and trim spaces
        text = text.strip().lower()

        # Convert Bengali numerals to words
        bangla_digits = {
            "১": "এক", "২": "দুই", "৩": "তিন", "৪": "চার", "৫": "পাঁচ",
            "৬": "ছয়", "৭": "সাত", "৮": "আট", "৯": "নয়", "০": "শূন্য"
        }
        for digit, word in bangla_digits.items():
            text = text.replace(digit, word)

        # Remove dাড়ি (।), double dাড়ি, symbols, and spaces
        text = re.sub(r'[\u0964\u0965\u09f3\s]', '', text)

        # Remove common classifiers/suffixes
        suffixes = ["টি", "জন", "অটোরিকশা", "টা", "টিই", "জনই"]
        for suffix in suffixes:
            text = text.replace(suffix, "")

        return text

    # ==========================================
    # 2. REFINED ACCURACY LOGIC
    # ==========================================
    def check_refined_accuracy(row):
        gt_raw = str(row['answer']).strip()
        gt = normalize_text(row['answer'])
        pred = normalize_text(row['prediction'])

        # Rule 1: Complete normalized exact match
        if gt == pred:
            return 1

        # Rule 2: Guardrail for strict boolean existence answers ("হ্যাঁ" / "না")
        if gt_raw in ["হ্যাঁ", "না"]:
            return 1 if gt == pred else 0

        # Rule 3: Sub-string fallback match for descriptive/open-ended answers
        if gt in pred or pred in gt:
            return 1

        return 0

    # Initialize list for tracking master report data
    all_summaries = []

    # ==========================================
    # 3. BATCH PROCESSING LOOP
    # ==========================================
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        model_name = os.path.splitext(file_name)

        print(f"==========================================")
        print(f"Processing Model: {model_name}")
        print(f"==========================================")

        # Load data
        df = pd.read_csv(file_path)

        # Ensure column fallback alignment (answer vs ground_truth)
        if 'answer' not in df.columns and 'ground_truth' in df.columns:
            df['answer'] = df['ground_truth']

        # Standard clean strings
        df['answer'] = df['answer'].astype(str).str.strip()
        df['prediction'] = df['prediction'].astype(str).str.strip()

        # Compute row-level accuracy metrics
        df['strict_exact_match'] = (df['answer'] == df['prediction']).astype(int)
        df['refined_accuracy'] = df.apply(check_refined_accuracy, axis=1)

        # Micro Overall Metrics
        overall_strict_em = df['strict_exact_match'].mean() * 100
        overall_refined = df['refined_accuracy'].mean() * 100

        # Category-wise metrics using Refined Accuracy logic for breakdowns
        category_refined_acc = df.groupby('q_type')['refined_accuracy'].mean() * 100
        macro_avg_refined = category_refined_acc.mean()

        # BERTScore (Semantic Similarity)
        print("Calculating BERTScore...")
        P, R, F1 = score(df['prediction'].tolist(), df['answer'].tolist(), lang="bn", verbose=False)
        df['bert_score_f1'] = F1.numpy()
        avg_bert = F1.mean().item() * 100

        # Save the full comprehensive metric sheet for this specific model
        detailed_output_path = os.path.join(OUTPUT_FOLDER, f"{model_name}_detailed_metrics.csv")
        df.to_csv(detailed_output_path, index=False, encoding='utf-8-sig')
        print(f" Saved detailed metrics to: {detailed_output_path}")

        # Create structured summary entry for the model row
        summary_data = {
            "Model Name": model_name,
            "Total Predictions": len(df),
            "Strict Exact Match (%)": round(overall_strict_em, 2),
            "Overall Refined Accuracy (%)": round(overall_refined, 2),
            "Macro Avg Refined Acc (%)": round(macro_avg_refined, 2),
            "Average BERTScore F1 (%)": round(avg_bert, 2)
        }

        # Append category-wise refined metrics dynamically
        for cat, acc in category_refined_acc.items():
            summary_data[f"Refined Acc: {cat} (%)"] = round(acc, 2)

        all_summaries.append(summary_data)
        print(f" Finished {model_name}.\n")

    # ==========================================
    # 4. SAVE AND EXPORT MASTER REPORT
    # ==========================================
    summary_df = pd.DataFrame(all_summaries)
    master_summary_path = os.path.join(OUTPUT_FOLDER, "overall_models_summary_report.csv")
    summary_df.to_csv(master_summary_path, index=False, encoding='utf-8-sig')

    print("==========================================")
    print(f" ALL MODELS PROCESSED SUCCESSFULLY!")
    print(f"Master Summary Report saved at: {master_summary_path}")
    print("==========================================")

    # Output matrix visually to the Colab screen
    summary_df

In [ ]:
import pandas as pd

# 2. Define File Paths (Adjust folder and file names as needed)
input_file_path = '/content/drive/MyDrive/Research/BVQA Dataset/BVQA_Master/3_Annotations/Performance Records/Performance Matrices/On Test Split/final_overall_models_summary_report.csv'
output_file_path = '/content/drive/MyDrive/Research/BVQA Dataset/BVQA_Master/3_Annotations/Performance Records/Performance Matrices/On Test Split/levelWise_report.csv'

# Load the existing evaluation CSV
df = pd.read_csv(input_file_path)

# 3. Calculate Cognitive Tier Accuracies
# Level 1: Average of Counting and Existence
df['Refined Acc: Level 1 (Perception) (%)'] = df[['Refined Acc: Counting (%)', 'Refined Acc: Existence (%)']].mean(axis=1).round(2)

# Level 2: Directly mapped from Attribute
df['Refined Acc: Level 2 (Representation) (%)'] = df['Refined Acc: Attribute (%)'].round(2)

# Level 3: Average of OCR and Reasoning
df['Refined Acc: Level 3 (Cognition) (%)'] = df[['Refined Acc: OCR (%)', 'Refined Acc: Reasoning (%)']].mean(axis=1).round(2)

# 4. Reorder Columns for a Professional Layout
# This keeps your metadata first, followed by overall stats, then the new tier breakdowns
base_columns = [
    'Model Name', 'Total Predictions', 'Strict Exact Match (%)',
    'Overall Refined Accuracy (%)', 'Macro Avg Refined Acc (%)', 'Average BERTScore F1 (%)'
]
tier_columns = [
    'Refined Acc: Level 1 (Perception) (%)',
    'Refined Acc: Level 2 (Representation) (%)',
    'Refined Acc: Level 3 (Cognition) (%)'
]
original_breakdowns = [
    'Refined Acc: Attribute (%)', 'Refined Acc: Counting (%)',
    'Refined Acc: Existence (%)', 'Refined Acc: OCR (%)', 'Refined Acc: Reasoning (%)'
]

final_column_order = base_columns + tier_columns + original_breakdowns
df_final = df[final_column_order]

# 5. Save the updated results to a new CSV file
df_final.to_csv(output_file_path, index=False)

print("\n=== TIER-WISE ACCURACY PROCESSING COMPLETE ===")
print(f"Successfully processed {len(df_final)} models.")
print(f"New file saved at: {output_file_path}")

# Preview the new columns for the first few rows
print("\nPreview of generated tier metrics:")
print(df_final[['Model Name'] + tier_columns].head())

In [ ]:
import os, glob, re, unicodedata
import pandas as pd

ROOT = "/sessions/great-stoic-planck/mnt/On Test Split"

# ============================================================
# CORRECTED normalize_text
#   fixes vs original:
#   - NFC-normalize first (kills encoding-variant false errors)
#   - map Arabic digits 0-9 too (original only mapped Bengali ১-৯)
#   - REMOVE "অটোরিকশা" from suffix list (it wiped whole answers)
#   - anchor classifier suffixes to END of string (endswith), not anywhere
# ============================================================
BANGLA_DIGITS = {"১":"এক","২":"দুই","৩":"তিন","৪":"চার","৫":"পাঁচ",
                 "৬":"ছয়","৭":"সাত","৮":"আট","৯":"নয়","০":"শূন্য"}
ARABIC_DIGITS = {"0":"শূন্য","1":"এক","2":"দুই","3":"তিন","4":"চার",
                 "5":"পাঁচ","6":"ছয়","7":"সাত","8":"আট","9":"নয়"}
# classifier/quantifier suffixes, longest-first so "টিই" strips before "টি"
END_SUFFIXES = ["জনই","টিই","জন","টি","টা"]

def normalize_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize("NFC", text)          # FIX: encoding variants
    text = text.strip().lower()
    for d, w in BANGLA_DIGITS.items():
        text = text.replace(d, w)
    for d, w in ARABIC_DIGITS.items():                 # FIX: Arabic numerals
        text = text.replace(d, w)
    text = re.sub(r'[।॥৳\s]', '', text)  # dari, double-dari, ৳, whitespace
    # FIX: strip a trailing classifier only (endswith-anchored), once
    for suf in END_SUFFIXES:
        if text.endswith(suf) and len(text) > len(suf):
            text = text[: -len(suf)]
            break
    return text

def check_refined(ans, pred):
    gt_raw = unicodedata.normalize("NFC", str(ans)).strip()
    gt = normalize_text(ans)
    pr = normalize_text(pred)
    # FIX: empty on either side is never a correct match
    if gt == "" or pr == "":
        return 0
    if gt == pr:
        return 1
    if gt_raw in ["হ্যাঁ", "না"]:
        return 1 if gt == pr else 0
    if gt in pr or pr in gt:
        return 1
    return 0

# ---- optional: order-insensitive credit for multi-part set answers ----
SEP = re.compile(r'[,/]|ও|এবং|-|;')
def tokset(text):
    parts = SEP.split(unicodedata.normalize("NFC", str(text)))
    return frozenset(t for t in (normalize_text(p) for p in parts) if t)

def check_refined_setaware(ans, pred):
    base = check_refined(ans, pred)
    if base == 1:
        return 1
    gt_raw = unicodedata.normalize("NFC", str(ans)).strip()
    if gt_raw in ["হ্যাঁ", "না"]:
        return 0
    gs, ps = tokset(ans), tokset(pred)
    # credit only when GT is genuinely multi-part and the sets match exactly
    if len(gs) > 1 and gs == ps:
        return 1
    return 0

# ---------------- name mapping (same as before) ----------------
NAMES = {
 "gpt-5.5_fewshot":"gpt-5.5_fewshot","claude-opus-4.7_few-shot":"claude-opus-4.7_few-shot",
 "gemini-3.5-flash_zero-shot":"gemini-3.5-flash_zero-shot","gemini-3.5-flash_few-shot":"gemini-3.5-flash_few-shot",
 "gemini-2.5-pro_zero-shot":"gemini-2.5-pro_zero-shot","gemini-2.5-pro_few-shot":"gemini-2.5-pro_few-shot",
 "gemini-2.5-flash_zero-shot":"gemini-2.5-flash_zero-shot","gemini-2.5-flash_few-shot":"gemini-2.5-flash_few-shot",
 "llama-3.2-vision-11b_zero_shot":"llama-3.2-vision-11b_zero_shot","llama-3.2-vision-11b_few_shot":"llama-3.2-vision-11b_few_shot",
 "internvl2_5_8b_zeroshot_final":"internvl2_5_8b_zeroshot","internvl2_5_8b_fewshot_final":"internvl2_5_8b_fewshot",
 "qwen25_vl_7b_zeroshot_test":"qwen25_vl_7b_zeroshot","qwen25_vl_7b_fewshot_test":"qwen25_vl_7b_fewshot",
 "pangea_7b_zeroshot":"pangea_7b_zeroshot","pangea_7b_fewshot":"pangea_7b_fewshot",
 "smolvlm2_base_model_zeroshot":"smolvlm2_base_model_zeroshot","smolvlm2_base_model_few_shot_test":"smolvlm2_base_model_few_shot",
 "smolvlm2_fineTuned_test":"smolvlm2_fineTuned","BBert_ViT_fineTuned":"BBert_ViT_fineTuned",
 "BBert_ResNet_fineTuned":"Bbert_ResNet50_fineTuned",
}
def report_name(p):
    return NAMES.get(os.path.splitext(os.path.basename(p))[0], os.path.splitext(os.path.basename(p))[0])

# level map
L1 = {"Existence","Counting"}; L2 = {"Attribute"}; L3 = {"Reasoning","OCR"}
def level(t): return "L1" if t in L1 else ("L2" if t in L2 else "L3")

# reported RA for delta
REPORTED = {
 "gpt-5.5_fewshot":83.54,"claude-opus-4.7_few-shot":82.72,"gemini-3.5-flash_zero-shot":80.67,
 "gemini-3.5-flash_few-shot":82.13,"gemini-2.5-pro_zero-shot":75.07,"gemini-2.5-pro_few-shot":76.75,
 "gemini-2.5-flash_zero-shot":74.68,"gemini-2.5-flash_few-shot":75.27,"llama-3.2-vision-11b_zero_shot":17.28,
 "llama-3.2-vision-11b_few_shot":25.07,"internvl2_5_8b_zeroshot":9.21,"internvl2_5_8b_fewshot":20.41,
 "qwen25_vl_7b_zeroshot":26.49,"qwen25_vl_7b_fewshot":37.11,"pangea_7b_zeroshot":17.0,"pangea_7b_fewshot":38.81,
 "smolvlm2_base_model_zeroshot":1.84,"smolvlm2_base_model_few_shot":4.25,"smolvlm2_fineTuned":46.88,
 "BBert_ViT_fineTuned":60.62,"Bbert_ResNet50_fineTuned":53.12,
}

files = sorted(glob.glob(os.path.join(ROOT,"**/*.csv"), recursive=True))
files = [f for f in files if os.path.basename(f) not in
         ("levelWise_report.csv","final_overall_models_summary_report.csv","overall_models_summary_report.csv")]

rows=[]; long=[]
for f in files:
    df = pd.read_csv(f)
    df.columns=[c.lstrip('﻿') for c in df.columns]
    if 'answer' not in df.columns and 'ground_truth' in df.columns: df['answer']=df['ground_truth']
    df['answer']=df['answer'].astype(str).str.strip()
    df['prediction']=df['prediction'].astype(str).str.strip()
    df['EM']=(df['answer']==df['prediction']).astype(int)
    df['RA_fix']   = df.apply(lambda r: check_refined(r['answer'], r['prediction']), axis=1)
    df['RA_setaware'] = df.apply(lambda r: check_refined_setaware(r['answer'], r['prediction']), axis=1)
    df['level']=df['q_type'].apply(level)
    name=report_name(f)
    def levra(col):
        return {lv: round(df[df.level==lv][col].mean()*100,2) for lv in ["L1","L2","L3"]}
    lf=levra('RA_fix')
    rep=REPORTED.get(name)
    rafix=round(df['RA_fix'].mean()*100,2)
    raset=round(df['RA_setaware'].mean()*100,2)
    rows.append({"model":name,"n":len(df),"EM":round(df['EM'].mean()*100,2),
                 "RA_reported":rep,"RA_fix":rafix,"d_fix":round(rafix-rep,2) if rep else None,
                 "RA_setaware":raset,"d_set":round(raset-rep,2) if rep else None,
                 "L1_fix":lf["L1"],"L2_fix":lf["L2"],"L3_fix":lf["L3"]})
    t=df.copy(); t['model']=name
    long.append(t[['model','image_id','q_type','level','question','answer','prediction','EM','RA_fix','RA_setaware']])

out=pd.DataFrame(rows).sort_values('RA_fix',ascending=False)
pd.concat(long,ignore_index=True).to_csv("rescored_all.csv",index=False,encoding='utf-8-sig')
out.to_csv("rescore_summary.csv",index=False,encoding='utf-8-sig')
pd.set_option('display.width',240); pd.set_option('display.max_columns',20)
print(out.to_string(index=False))